In [ ]:
# ================================================================
# USER CONFIGURATION — set these paths for your environment
# ================================================================

# Root directory for saving anomaly maps — change to your Drive path
# Maps are saved under: {MAPS_SAVE_DIR}/crossview/{model_name}/{category}/
MAPS_SAVE_DIR = ''          # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# Cross-Viewpoint Protocol — Real-IAD Robustness Benchmark

Evaluates model robustness to unseen camera viewpoints.
Models are trained on viewpoints C1 and C2 only, then evaluated on
the unseen viewpoints C3, C4, and C5.

This protocol is novel for all three models and directly addresses
the research question on viewpoint robustness in industrial inspection.

Models: Dinomaly, INP-Former, AnomalyDINO
Dataset: Real-IAD 512px, 30 categories
Reference: Standard protocol results from 02_standard_protocol.ipynb

In [ ]:
# Reduce CUDA memory fragmentation
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

# Clone repo if not already present, otherwise pull latest
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

# Force INP-Former submodule to correct commit with path fixes
!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

# Install dependencies
!pip install anomalib==2.4.0 ADEval einops colorama timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Results paths — consistent across all notebooks
results_path = f'{repo_path}/results'

# Protocol-specific score paths
std_results = f'{results_path}/standard'
cv_results = f'{results_path}/crossview'
abl_results = f'{results_path}/ablation'

# Anomaly map paths
std_maps = f'{MAPS_SAVE_DIR}/standard'
cv_maps = f'{MAPS_SAVE_DIR}/crossview'
maps_abl1_mm = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_multiview'
maps_abl1_sm = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_multiview'
maps_abl1_ms = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_singleview'
maps_abl1_ss = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_singleview'
maps_abl2 = f'{MAPS_SAVE_DIR}/ablation/investigation2'
maps_abl3 = f'{MAPS_SAVE_DIR}/ablation/investigation3'

# Create all directories upfront
for path in [
    std_results,
    cv_results,
    f'{abl_results}/investigation1',
    f'{abl_results}/investigation2',
    f'{abl_results}/investigation3',
    f'{results_path}/weights',
    f'{results_path}/figures',
    std_maps,
    cv_maps,
    maps_abl1_mm,
    maps_abl1_sm,
    maps_abl1_ms,
    maps_abl1_ss,
    maps_abl2,
    maps_abl3,
]:
    os.makedirs(path, exist_ok=True)

print("All results directories ready")

In [ ]:
import zipfile, os

zip_dir = '/content/drive/MyDrive/datasets/realiad_512/realiad_512'
target_dir = '/content/realiad_512'
os.makedirs(target_dir, exist_ok=True)

# Also copy JSON files
import shutil
json_src = '/content/drive/MyDrive/datasets/realiad_512/realiad_jsons'
json_dst = '/content/realiad_512/realiad_jsons'
if not os.path.exists(json_dst):
    shutil.copytree(json_src, json_dst)
    print("JSONs copied")

# Unzip each category
for f in sorted(os.listdir(zip_dir)):
    if f.endswith('.zip'):
        category = f.replace('.zip', '')
        if not os.path.exists(f'{target_dir}/{category}'):
            print(f"Unzipping {f}...")
            with zipfile.ZipFile(f'{zip_dir}/{f}', 'r') as z:
                z.extractall(target_dir)
        else:
            print(f"Skipping {category} — already exists")

print("Done")
dataset_root = target_dir
print(f"Active dataset root: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all = realiad_utils.load_realiad_all

# Dinomaly — now uses official repo via run_inference_dinomaly
train_dinomaly          = trainer.train_dinomaly
run_inference_dinomaly  = trainer.run_inference_dinomaly

# AnomalyDINO — 16-shot multi-class few-shot protocol
train_anomalydino_fewshot = trainer.train_anomalydino_fewshot

# INP-Former
train_inpformer         = trainer.train_inpformer
run_inference_inpformer = trainer.run_inference_inpformer

# Shared inference for AnomalyDINO
run_inference           = trainer.run_inference

# Utility functions
measure_inference_time  = trainer.measure_inference_time
measure_memory_footprint = trainer.measure_memory_footprint
get_crossview_split = realiad_utils.get_crossview_split

# Metrics
compute_i_auroc   = metrics.compute_i_auroc
compute_s_auroc   = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
REALIAD_CONFIG    = metrics.REALIAD_CONFIG

print("All modules loaded")

## Step 1: Load Real-IAD and Apply Cross-Viewpoint Split

Training is restricted to viewpoints C1 and C2.
Evaluation is performed on all viewpoints.
This simulates a deployment scenario where the model is commissioned
on a limited set of camera angles and must generalise to new viewpoints.

In [ ]:
# Load all 30 categories
df = load_realiad_all(data_root=dataset_root)

# Apply cross-viewpoint split
train_views = ['C1', 'C2']
test_views = ['C1', 'C2', 'C3', 'C4', 'C5']

train_df_cv, test_df_cv = get_crossview_split(
    df,
    train_views=train_views,
    test_views=test_views
)

# Training set: normal images from C1 and C2 only
train_df_cv = train_df_cv[train_df_cv['label'] == 0].reset_index(drop=True)

print(f"Train views: {train_views}")
print(f"Test views: {test_views}")
print(f"Train (normal only): {len(train_df_cv)}")
print(f"Test total: {len(test_df_cv)}")
print(f"Test label distribution:\n{test_df_cv['label'].value_counts()}")

os.makedirs(f'{repo_path}/results', exist_ok=True)
df.to_csv(f'{repo_path}/results/dataset_split_crossview.csv', index=False)
print("Dataset split saved")

## Step 2: Dinomaly — Cross-Viewpoint Protocol

Same architecture and training configuration as standard protocol.
Only the training data differs: C1+C2 viewpoints instead of all five.

In [ ]:
model_dinomaly_cv = train_dinomaly(
    train_df=train_df_cv,
    n_iterations=50000,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/dinomaly_crossview.pth'
)

results_din_cv = run_inference_dinomaly(
    model=model_dinomaly_cv,
    test_df=test_df_cv,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=cv_maps
)

results_din_cv.to_csv(
    f'{cv_results}/dinomaly_scores.csv', index=False)
print(f"Dinomaly cross-view saved: {len(results_din_cv)} rows")
print(f"I-AUROC: {compute_i_auroc(results_din_cv):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_dinomaly_cv
print("GPU memory cleared")

## Step 3: INP-Former — Cross-Viewpoint Protocol

Prototype tokens are learned from C1+C2 normal features only.
At inference time the prototypes must generalise to the unseen C3+C4+C5 viewpoints.

In [ ]:
model_inpformer_cv = train_inpformer(
    train_df=train_df_cv,
    dataset_root=dataset_root,
    n_epochs=100,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_crossview.pth'
)

results_inp_cv = run_inference_inpformer(
    model=model_inpformer_cv,
    test_df=test_df_cv,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=cv_maps
)

results_inp_cv.to_csv(
    f'{cv_results}/inpformer_scores.csv', index=False)
print(f"INP-Former cross-view saved: {len(results_inp_cv)} rows")
print(f"I-AUROC: {compute_i_auroc(results_inp_cv):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inpformer_cv
print("GPU memory cleared")

## Step 4: AnomalyDINO — Cross-Viewpoint Protocol

Memory bank built from C1+C2 viewpoints only (16 shots x 30 categories
x 2 viewpoints = 960 reference images). No gradient-based training.
Inference runs on the full test set across all five viewpoints.

In [ ]:
# AnomalyDINO — Cross-Viewpoint Protocol



model_ad_cv = train_anomalydino_fewshot(
    train_df=train_df_cv,
    n_shots=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/anomalydino_cv.pth'
)

results_anomalydino_cv = run_inference(
    model=model_ad_cv,
    test_df=test_df_cv,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=cv_maps
)

os.makedirs(cv_results, exist_ok=True)
results_anomalydino_cv.to_csv(
    f'{cv_results}/anomalydino_scores.csv', index=False)

i_auroc_ad_cv = compute_i_auroc(results_anomalydino_cv)
print(f'AnomalyDINO cross-view I-AUROC: {i_auroc_ad_cv:.4f}')

del model_ad_cv
torch.cuda.empty_cache()
gc.collect()

## Step 5: Performance Degradation Analysis

Compares cross-viewpoint results against standard protocol results.
The degradation ratio quantifies sensitivity to viewpoint shift.
A higher ratio indicates greater dependence on training viewpoint coverage.

In [ ]:
# Load standard protocol results for comparison
results_dinomaly_std  = pd.read_csv(f'{std_results}/dinomaly_scores.csv')
results_inpformer_std = pd.read_csv(f'{std_results}/inpformer_scores.csv')
results_anomalydino_std = pd.read_csv(
    f'{std_results}/anomalydino_scores.csv')

i_auroc_din_std  = compute_i_auroc(results_dinomaly_std)
i_auroc_inp_std  = compute_i_auroc(results_inpformer_std)
i_auroc_ad_std   = compute_i_auroc(results_anomalydino_std)

deg_dinomaly   = compute_degradation_ratio(i_auroc_din_std, i_auroc_din_cv)
deg_inpformer  = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_cv)
deg_anomalydino = compute_degradation_ratio(i_auroc_ad_std,  i_auroc_ad_cv)

summary = pd.DataFrame({
    'Model': ['Dinomaly', 'INP-Former', 'AnomalyDINO'],
    'I-AUROC Standard':   [i_auroc_din_std,  i_auroc_inp_std,  i_auroc_ad_std],
    'I-AUROC Cross-View': [i_auroc_din_cv,   i_auroc_inp_cv,   i_auroc_ad_cv],
    'Degradation (%)':    [deg_dinomaly,     deg_inpformer,    deg_anomalydino],
})

print("=" * 60)
print("CROSS-VIEWPOINT PROTOCOL RESULTS")
print("=" * 60)
print(summary.round(4).to_string(index=False))
summary.to_csv(f'{cv_results}/summary.csv', index=False)
print(f"\nResults saved to crossview/summary.csv")

wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
print_wga_summary = wga_module.print_wga_summary
df_dict_cv = {
    'Dinomaly':    results_dinomaly_cv,
    'INP-Former':  results_inpformer_cv,
    'AnomalyDINO': results_anomalydino_cv
}
print_wga_summary(df_dict_cv)